# M2 · Your First Agent

> **Goal:** turn a raw model into a **named, versioned agent** — give it instructions, create it on Foundry, invoke it, then iterate safely.
> **You'll use:** `PromptAgentDefinition`, `project_client.agents.create_version`, the Responses API with an `agent_reference`.

---

In [M1](../01-first-inference/) you called a model directly. An
**agent** wraps that model in a reusable, server-side definition:

> **agent = model + instructions + tools**

The definition lives in your Foundry project under a stable **name**. Each time you
change it, Foundry stores a new **version** — so you can iterate on a prompt without
ever breaking the callers that reference the agent by name.

![Anatomy of a Foundry agent](../../assets/agent-anatomy.png)

If your project and `.env` aren't ready yet, do the [Setup](../../setup/) first.

In [1]:
# print current date and time
from datetime import datetime

# Get the current date and time
current_datetime = datetime.now()

# Print the current date and time
print("Current date and time:", current_datetime)

Current date and time: 2026-08-27 10:11:52.194276


## 1. Configure

Same `.env` as every lab (see [Setup](../../setup/)). We read the
project endpoint and the chat model deployment, and pick a stable agent name.

In [2]:
import os
from dotenv import load_dotenv

load_dotenv()  # reads .env from the repo root

PROJECT_ENDPOINT = os.environ["PROJECT_ENDPOINT"]
CHAT_MODEL       = os.environ.get("CHAT_MODEL", "gpt-4.1-mini")

# A stable, human-readable name. Re-running these cells versions THIS agent.
AGENT_NAME = "storytelling-agent"
# Only print first 15 characters of project endpoint for security reasons
print("Project :", PROJECT_ENDPOINT[:15])
print("Chat    :", CHAT_MODEL)
print("Agent   :", AGENT_NAME)

Project : https://aibslab
Chat    : gpt-4.1-mini
Agent   : storytelling-agent


!!! note "Expected output"
    ```
    Project : https://<account>.services.ai.azure.com/api/projects/<project>
    Chat    : gpt-4.1-mini
    Agent   : storytelling-agent
    ```
    The agent name is yours to choose — keep it stable, since versioning keys off it.

## 2. Build the client

Identical bootstrap to M1: `DefaultAzureCredential` → `AIProjectClient` → an
OpenAI-compatible client. We also reach `project_client.agents`, the surface for
creating and versioning agents.

In [3]:
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient

credential     = DefaultAzureCredential()
project_client = AIProjectClient(endpoint=PROJECT_ENDPOINT, credential=credential)
openai_client  = project_client.get_openai_client()

print("project_client : ready")
print("openai_client  : ready")

project_client : ready
openai_client  : ready


!!! note "Expected output"
    ```
    project_client : ready
    openai_client  : ready
    ```
    A credential error here usually means you need `az login`; a `403` means your
    identity lacks the **Azure AI Developer** role on the project.

## 3. Define and create the agent

A `PromptAgentDefinition` is the whole agent: the **model** it runs on and the
**instructions** (system prompt) that shape its behaviour. (Tools come in
[M3](../03-tools-and-function-calling/).) You hand that
definition to `create_version`, which stores it under your chosen name.

In [4]:
from azure.ai.projects.models import PromptAgentDefinition

agent = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_MODEL,
        instructions=(
            "You are a storytelling agent. "
            "You craft engaging one-line stories based on user prompts and context."
        ),
    ),
)

print("Name    :", agent.name)
print("Version :", agent.version)

Name    : storytelling-agent
Version : 8


!!! note "Expected output"
    ```
    Name    : storytelling-agent
    Version : 1
    ```

!!! tip "`create_version` is idempotent"
    Re-running this exact cell does **not** spawn version 2 — Foundry compares the
    definition to the latest stored version and only bumps the number when something
    actually changed. That makes it safe to re-run while you iterate.

## 4. Invoke the agent

You call the agent through the same `responses.create(...)` surface from M1 — but
instead of passing `model=`, you attach an **`agent_reference`** in `extra_body`.
Foundry resolves the name, applies the stored model + instructions, and returns the
reply in `output_text`.

In [5]:
response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me a one-line story about a lighthouse keeper."}],
    extra_body={"agent_reference": {"name": agent.name, "type": "agent_reference"}},
)

print(response.output_text)

Every night, the lighthouse keeper whispered lost sailors’ names to the wind, guiding their spirits safely home.


!!! note "Expected output"
    ```
    Every night the keeper lit the lamp for ships that never came — until the night
    one finally did, carrying the letter he'd stopped waiting for.
    ```
    Wording varies run to run; what matters is that the model now speaks in the
    *voice* your instructions defined, without you resending the system prompt.

## 5. Version the agent

This is the payoff. Change the **instructions** and call `create_version` again — same
name, new version. Existing callers keep working; you've simply published a new
revision they can pick up. Here we make the agent gloomier.

In [6]:
agent_v2 = project_client.agents.create_version(
    agent_name=AGENT_NAME,                       # same name → new version
    definition=PromptAgentDefinition(
        model=CHAT_MODEL,
        instructions=(
            "You are a storytelling agent with a melancholic, noir voice. "
            "You craft a single haunting sentence based on the user's prompt."
        ),
    ),
)

print("Name    :", agent_v2.name)
print("Version :", agent_v2.version)   # incremented because instructions changed

response = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me a one-line story about a lighthouse keeper."}],
    extra_body={"agent_reference": {"name": agent_v2.name, "type": "agent_reference"}},
)
print()
print(response.output_text)

Name    : storytelling-agent
Version : 9



Amid the relentless storm, the lighthouse keeper’s final light flickered out, leaving only the cold, dark sea to remember his silent sacrifice.


!!! note "Expected output"
    ```
    Name    : storytelling-agent
    Version : 2

    The lamp still turns, but the keeper stopped counting the years the sea kept
    taking from him.
    ```

!!! warning "Name stays, version moves"
    The **name** is the stable contract callers depend on; the **version** is the
    audit trail of how the agent evolved. Never rename to iterate — re-version.

## 🧪 Your turn

1. **Reshape the voice.** Rewrite the instructions in section 5 (e.g. *cheerful
   children's-book narrator*) and re-run. Confirm the version increments and the tone
   flips.
2. **Prove idempotency.** Run the *unchanged* section 3 cell twice in a row and watch
   `agent.version` hold steady — then change a single word and watch it bump.
3. **Give it context.** Add a second message to `input` (a `system`-style preface or a
   prior turn) and see how the agent blends your per-call context with its stored
   instructions.

---

✅ **You created a named agent, invoked it via `agent_reference`, and versioned it
safely.** Next: give your agent real **tools** — code execution and your own functions.
→ **[M3 · Tools & Function Calling](../03-tools-and-function-calling/)**

## ✅ Your turn — solutions

The cells below complete the three **Your turn** challenges. Run the notebook
top-to-bottom first so `project_client`, `openai_client`, `AGENT_NAME`, and
`CHAT_MODEL` exist.

### 1 · Reshape the voice

Re-version the agent with a **cheerful children's-book narrator** voice (section 5
made it noir). The version increments because the instructions changed, and the tone
flips.

In [7]:
from azure.ai.projects.models import PromptAgentDefinition

cheerful = project_client.agents.create_version(
    agent_name=AGENT_NAME,
    definition=PromptAgentDefinition(
        model=CHAT_MODEL,
        instructions=(
            "You are a cheerful children's-book narrator. You tell a single warm, "
            "playful sentence full of wonder, based on the user's prompt."
        ),
    ),
)
print("Name    :", cheerful.name)
print("Version :", cheerful.version, "(bumped — instructions changed)")

resp = openai_client.responses.create(
    input=[{"role": "user", "content": "Tell me a one-line story about a lighthouse keeper."}],
    extra_body={"agent_reference": {"name": cheerful.name, "version": cheerful.version,
                                    "type": "agent_reference"}},
)
print("\n" + resp.output_text)

Name    : storytelling-agent
Version : 10 (bumped — instructions changed)



Every night, the cheerful lighthouse keeper sang to the stars, guiding lost ships with a twinkle in his eye and a song in his heart.


### 2 · Prove idempotency

Creating a version with the **same** definition does *not* mint a new version — the
version number holds steady. Change a single word and it bumps.

In [8]:
def version_for(instructions):
    v = project_client.agents.create_version(
        agent_name=AGENT_NAME,
        definition=PromptAgentDefinition(model=CHAT_MODEL, instructions=instructions),
    )
    return v.version

base = ("You are a storytelling agent with a melancholic, noir voice. "
        "You craft a single haunting sentence based on the user's prompt.")

first  = version_for(base)
second = version_for(base)                       # identical definition → no new version
print(f"Unchanged re-run : {first} then {second}   -> steady: {first == second}")

third = version_for(base + " Always name the sea.")   # one word changed → bump
print(f"After 1-word edit: {third}            -> bumped: {third != first}")

Unchanged re-run : 11 then 11   -> steady: True


After 1-word edit: 12            -> bumped: True


### 3 · Give it context

Blend **per-call context** with the agent's stored instructions. The Responses API +
`agent_reference` doesn't accept a `system` role here, so we supply the context as a
**prior turn** (a user/assistant exchange) ahead of the real question.

In [9]:
resp = openai_client.responses.create(
    input=[
        {"role": "user", "content": "For all our stories, the hero is a small brave crab named Pip."},
        {"role": "assistant", "content": "Understood — Pip the crab is our hero."},
        {"role": "user", "content": "Tell me a one-line story about a lighthouse keeper."},
    ],
    extra_body={"agent_reference": {"name": cheerful.name, "version": cheerful.version,
                                    "type": "agent_reference"}},
)
print(resp.output_text)

Every night, the lighthouse keeper whispered stories to the stars, lighting up the sky with dreams and guiding ships safely home.
